In [52]:
import pandas as pd

# read data
df = pd.read_csv("../data/dirty/ILO/EAR_4MTH_SEX_CUR_NB_A-20250408T0015.csv")

# regex
df["sex"] = df["sex.label"].str.extract(r"Sex: (\w)")[0]
df["ISO"] = df["note_indicator.label"].str.extract(r"Currency: (\w{3})")[0]
df["currency"] = df["classif1.label"].str.extract(r"Currency: (.+)")[0]
df["local_currency_code"] = (
    df["note_indicator.label"]
    .str.extract(r"Currency:\s*\w{3}\s*-\s*[^()]*\(\s*([A-Z]{3})\s*\)")[0]
    .str.strip()
)
df["break"] = df["note_indicator.label"].str.contains("Break in series", na=False)
# remove irrelevant data
df.drop(
    columns=[
        "obs_status.label",
        "note_classif.label",
        "note_source.label",
        "indicator.label",
        "note_indicator.label",
        "source.label",
        "sex.label",
        "classif1.label",
    ],
    inplace=True,
)

# rename columns
df.rename(
    columns={
        "ref_area.label": "country",
        "time": "year",
        "obs_value": "wage",
        "ISO": "ISO",
    },
    inplace=True,
)

df["year"] = pd.to_numeric(df["year"], errors="coerce")

# index max year per country and sex
idx = df.groupby(["country", "sex", "currency"])["year"].idxmax()
# df = df.loc[idx].reset_index(drop=True)

# last_break = df[df["break"]].groupby("ISO")["year"].max().rename("last_break")
# df = df.merge(last_break, on="ISO", how="left")
# df["last_break"] = pd.to_numeric(df["last_break"], errors="coerce")

map = {"U.S. dollars": "USD", "2021 PPP $": "PPP", "Local currency": "Local"}
df["currency"] = df["currency"].replace(map)

df.to_csv("../data/clean/ILO.csv")

In [53]:
df_ratio = df.pivot_table(
    index=["ISO", "year", "country", "currency", "local_currency_code"],
    columns="sex",
    values="wage",
    aggfunc="first",
).reset_index()

break_info = df[
    [
        "ISO",
        "year",
        "break",
        # "last_break",
    ]
].drop_duplicates(subset=["ISO", "year"])

df_ratio = df_ratio.merge(break_info, on=["ISO", "year"], how="left")

df_ratio["ratio"] = df_ratio["F"] / df_ratio["M"]

df_ratio = df_ratio[
    [
        "ISO",
        "year",
        "country",
        "T",
        "M",
        "F",
        "ratio",
        "break",
        "local_currency_code",
        # "last_break",
        "currency",
    ]
]
df_ratio

,ISO,year,country,T,M,F,ratio,break,local_currency_code,currency
0,ABW,2010,Aruba,3013.000,3338.000,2713.000,0.812762,False,AWG,Local
1,ABW,2010,Aruba,1860.382,2061.054,1675.147,0.812762,False,AWG,PPP
2,ABW,2010,Aruba,1683.240,1864.804,1515.642,0.812762,False,AWG,USD
3,AFG,2014,Afghanistan,8848.569,9135.149,5517.315,0.603966,False,AFN,Local
4,AFG,2014,Afghanistan,517.709,534.476,322.805,0.603965,False,AFN,PPP
...,...,...,...,...,...,...,...,...,...,...
8380,ZWE,2022,Zimbabwe,212.044,223.145,193.270,0.866118,False,ZWL,PPP
8381,ZWE,2022,Zimbabwe,80.864,85.097,73.704,0.866117,False,ZWL,USD
8382,ZWE,2023,Zimbabwe,406978.727,420431.180,385130.787,0.916038,False,ZWL,Local
8383,ZWE,2023,Zimbabwe,1711.902,1768.488,1620.001,0.916037,False,ZWL,PPP


In [72]:
df_ratio["female_share"] = (df_ratio["T"] - df_ratio["M"]) / (
    df_ratio["F"] - df_ratio["M"]
)
df_ratio["female_share"] = df_ratio["female_share"].clip(0, 1)
df_ratio["male_share"] = 1 - df_ratio["female_share"]

df_ratio["balanced_avg"] = (df_ratio["F"] + df_ratio["M"]) / 2
df_ratio["balanced_ratio"] = df_ratio["F"] / df_ratio["balanced_avg"]

df_ratio.to_csv("../data/clean/ILO_ratio.csv")

In [73]:
df_ratio = df_ratio.dropna(subset=["F", "M"]).reset_index(drop=True)

df_wide = df_ratio.pivot_table(
    index=[
        "ISO",
        "year",
        "country",
        "break",
        "local_currency_code",
    ],
    columns="currency",
    values=["F", "M", "T", "ratio"],
    aggfunc="first",
).reset_index()

df_wide.columns = [
    f"{metric}_{curr.lower()}" if curr else metric for metric, curr in df_wide.columns
]

shares = df_ratio[
    [
        "ISO",
        "year",
        "country",
        "female_share",
        "male_share",
        "balanced_avg",
        "balanced_ratio",
    ]
].drop_duplicates(subset=["ISO", "year", "country"])

df_wide = df_wide.merge(shares, on=["ISO", "year", "country"], how="left")

print(df_wide)

      ISO  year      country  break local_currency_code     F_local     F_ppp  \
0     ABW  2010        Aruba  False                 AWG    2713.000  1675.147   
1     AFG  2014  Afghanistan  False                 AFN    5517.315   322.805   
2     AFG  2020  Afghanistan   True                 AFN   10843.294   686.841   
3     AGO  2019       Angola   True                 AOA   58793.485   404.360   
4     AGO  2021       Angola  False                 AOA   69497.802   341.196   
...   ...   ...          ...    ...                 ...         ...       ...   
2407  ZWE  2011     Zimbabwe   True                 ZWL     204.576       NaN   
2408  ZWE  2014     Zimbabwe  False                 ZWL     283.740       NaN   
2409  ZWE  2021     Zimbabwe  False                 ZWL   14481.879   191.960   
2410  ZWE  2022     Zimbabwe  False                 ZWL   27635.835   193.270   
2411  ZWE  2023     Zimbabwe  False                 ZWL  385130.787  1620.001   

         F_usd     M_local 

In [82]:
from currency_converter import CurrencyConverter
from datetime import date
import numpy as np

c = CurrencyConverter()

code_map = {
    "ALK": "ALL",
}


def fill_usd_if_missing(row, metric):
    usd_col = f"{metric}_usd"
    local_col = f"{metric}_local"
    if not pd.isna(row[usd_col]):
        return row[usd_col]

    code = row["local_currency_code"]
    code = code_map.get(code, code)
    try:
        return c.convert(row[local_col], code, "USD", date(int(row["year"]), 1, 1))
    except Exception as e:
        print(e)
        return np.nan


for metric in ["F", "M", "T"]:
    df_wide[f"{metric}_usd"] = df_wide.apply(
        lambda r: fill_usd_if_missing(r, metric), axis=1
    )

df_wide["ratio_usd"] = df_wide["F_usd"] / df_wide["M_usd"]
df_wide.to_csv("../data/clean/ILO_ratio_wide.csv")

1969-01-01 not in AUD bounds 1999-01-04/2025-04-03
1970-01-01 not in AUD bounds 1999-01-04/2025-04-03
1971-01-01 not in AUD bounds 1999-01-04/2025-04-03
1972-01-01 not in AUD bounds 1999-01-04/2025-04-03
1973-01-01 not in AUD bounds 1999-01-04/2025-04-03
1974-01-01 not in AUD bounds 1999-01-04/2025-04-03
1975-01-01 not in AUD bounds 1999-01-04/2025-04-03
1976-01-01 not in AUD bounds 1999-01-04/2025-04-03
1977-01-01 not in AUD bounds 1999-01-04/2025-04-03
1978-01-01 not in AUD bounds 1999-01-04/2025-04-03
1979-01-01 not in AUD bounds 1999-01-04/2025-04-03
1998-01-01 not in GBP bounds 1999-01-04/2025-04-03
HKD has no rate for 2005-01-01
SZL is not a supported currency
SZL is not a supported currency
SZL is not a supported currency
SZL is not a supported currency
SZL is not a supported currency
SZL is not a supported currency
SZL is not a supported currency
SZL is not a supported currency
SZL is not a supported currency


In [75]:
url = "https://en.wikipedia.org/wiki/Historical_exchange_rates_of_Argentine_currency"
tables = pd.read_html(url)

for tbl in tables:
    if list(tbl.columns[:2]) == ["Year", "Jan"]:
        wiki_fx = tbl.copy()
        break

wiki_fx.columns = wiki_fx.columns.str.strip()
wiki_fx["Year"] = wiki_fx["Year"].astype(int)

df_rates = wiki_fx.melt(
    id_vars="Year",
    value_vars=[
        "Jan",
        "Feb",
        "Mar",
        "Apr",
        "May",
        "Jun",
        "Jul",
        "Aug",
        "Sep",
        "Oct",
        "Nov",
        "Dec",
    ],
    var_name="Month",
    value_name="ars_per_usd",
)

df_rates["Month"] = pd.to_datetime(df_rates["Month"], format="%b").dt.month
df_jan = df_rates[df_rates["Month"] == 1].copy()

# rename and compute
df_jan = df_jan.rename(columns={"Year": "year"})
df_jan["usd_per_ars"] = 1.0 / df_jan["ars_per_usd"]

# merge into wide df
df_wide = df_wide.merge(df_jan[["year", "usd_per_ars"]], on="year", how="left")
print(df_wide)

# fill in the ARS conversions
mask_ars = df_wide["local_currency_code"] == "ARS"
for metric in ["F", "M", "T"]:
    df_wide.loc[mask_ars, f"{metric}_usd"] = (
        df_wide.loc[mask_ars, f"{metric}_local"] * df_wide.loc[mask_ars, "usd_per_ars"]
    )

# recompute USD ratio
df_wide["ratio_usd"] = df_wide["F_usd"] / df_wide["M_usd"]

      ISO  year      country  break local_currency_code     F_local     F_ppp  \
0     ABW  2010        Aruba  False                 AWG    2713.000  1675.147   
1     AFG  2014  Afghanistan  False                 AFN    5517.315   322.805   
2     AFG  2020  Afghanistan   True                 AFN   10843.294   686.841   
3     AGO  2019       Angola   True                 AOA   58793.485   404.360   
4     AGO  2021       Angola  False                 AOA   69497.802   341.196   
...   ...   ...          ...    ...                 ...         ...       ...   
2407  ZWE  2011     Zimbabwe   True                 ZWL     204.576       NaN   
2408  ZWE  2014     Zimbabwe  False                 ZWL     283.740       NaN   
2409  ZWE  2021     Zimbabwe  False                 ZWL   14481.879   191.960   
2410  ZWE  2022     Zimbabwe  False                 ZWL   27635.835   193.270   
2411  ZWE  2023     Zimbabwe  False                 ZWL  385130.787  1620.001   

         F_usd     M_local 

In [76]:
# manually compute aud to usd lmao
aud_per_usd = {
    1950: 0.89286,
    1951: 0.89286,
    1952: 0.89286,
    1953: 0.89286,
    1954: 0.89286,
    1955: 0.89286,
    1956: 0.89286,
    1957: 0.89286,
    1958: 0.89286,
    1959: 0.89286,
    1960: 0.89286,
    1961: 0.89286,
    1962: 0.89286,
    1963: 0.89286,
    1964: 0.89286,
    1965: 0.89286,
    1966: 0.89286,
    1967: 0.89286,
    1968: 0.89286,
    1969: 0.89286,
    1970: 0.89286,
    1971: 0.88267,
    1972: 0.83870,
    1973: 0.70411,
    1974: 0.69667,
    1975: 0.76387,
    1976: 0.81828,
    1977: 0.90182,
    1978: 0.87366,
    1979: 0.89464,
    1980: 0.87824,
    1981: 0.87021,
    1982: 0.98586,
    1983: 1.11001,
    1984: 1.13952,
    1985: 1.43198,
    1986: 1.49597,
    1987: 1.42818,
    1988: 1.27991,
    1989: 1.26460,
    1990: 1.28106,
    1991: 1.28376,
    1992: 1.36165,
    1993: 1.47056,
    1994: 1.36775,
    1995: 1.34903,
    1996: 1.27786,
    1997: 1.34738,
    1998: 1.59183,
    1999: 1.54995,
    2000: 1.72483,
    2001: 1.93344,
    2002: 1.84056,
    2003: 1.54191,
    2004: 1.35975,
    2005: 1.30947,
    2006: 1.32797,
    2007: 1.19507,
    2008: 1.19218,
    2009: 1.28219,
    2010: 1.09016,
    2011: 0.96946,
    2012: 0.96580,
    2013: 1.03584,
    2014: 1.10936,
    2015: 1.33109,
    2016: 1.34521,
    2017: 1.30476,
    2018: 1.33841,
    2019: 1.43851,
    2020: 1.45309,
    2021: 1.33122,
    2022: 1.44166,
    2023: 1.50519,
}

df_wide["aud_per_usd"] = df_wide["year"].map(aud_per_usd)

mask = df_wide["local_currency_code"] == "AUD"
for metric in ["F", "M", "T"]:
    df_wide.loc[mask, f"{metric}_usd"] = (
        df_wide.loc[mask, f"{metric}_local"] / df_wide.loc[mask, "aud_per_usd"]
    )

df_wide["ratio_usd"] = df_wide["F_usd"] / df_wide["M_usd"]
df_wide = df_wide[
    [
        "ISO",
        "year",
        "country",
        "break",
        "local_currency_code",
        "F_local",
        # "F_ppp",
        "F_usd",
        "M_local",
        # "M_ppp",
        "M_usd",
        "T_local",
        # "T_ppp",
        "T_usd",
        # "ratio_local",
        # "ratio_ppp",
        "ratio_usd",
        "female_share",
        "male_share",
        "balanced_avg",
        "balanced_ratio",
    ]
]
df_wide.rename(
    columns={
        "ratio_usd": "ratio",
    },
    inplace=True,
)
df_wide.to_csv("../data/clean/ILO_ratio_wide.csv")

In [77]:
float_cols = df_wide.select_dtypes(include="float").columns
df_wide[float_cols] = np.trunc(df_wide[float_cols] * 1000) / 1000
df_wide = df_wide.dropna(subset=["F_usd"]).reset_index(drop=True)

df_wide.to_csv("../data/clean/ILO_ratio_wide_clean.csv")
df_wide

,ISO,year,country,break,local_currency_code,F_local,F_usd,M_local,M_usd,T_local,T_usd,ratio,female_share,male_share,balanced_avg,balanced_ratio
0,ABW,2010,Aruba,False,AWG,2713.000,1515.642,3338.000,1864.804,3013.000,1683.240,0.812,0.520,0.480,3025.500,0.896
1,AFG,2014,Afghanistan,False,AFN,5517.315,96.376,9135.149,159.573,8848.569,154.567,0.603,0.079,0.920,7326.232,0.753
2,AFG,2020,Afghanistan,True,AFN,10843.294,141.164,13439.387,174.961,13202.245,171.874,0.806,0.091,0.908,12141.340,0.893
3,AGO,2019,Angola,True,AOA,58793.485,161.155,92241.630,252.838,82234.658,225.408,0.637,0.299,0.700,75517.557,0.778
4,AGO,2021,Angola,False,AOA,69497.802,110.062,99293.136,157.248,90678.796,143.606,0.699,0.289,0.710,84395.469,0.823
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1968,ZMB,2022,Zambia,False,ZMW,3245.492,191.615,3535.564,208.741,3445.361,203.415,0.917,0.310,0.689,3390.528,0.957
1969,ZMB,2023,Zambia,False,ZMW,3203.494,158.494,3257.913,161.187,3239.976,160.299,0.983,0.329,0.670,3230.703,0.991
1970,ZWE,2021,Zimbabwe,False,ZWL,14481.879,163.540,13633.437,153.959,13935.047,157.365,1.062,0.355,0.644,14057.658,1.030
1971,ZWE,2022,Zimbabwe,False,ZWL,27635.835,73.704,31907.583,85.097,30320.301,80.864,0.866,0.371,0.628,29771.709,0.928


In [78]:
low_q, high_q = df_wide["ratio"].quantile([0.1, 0.9])
# print(df_wide)
df_mid10 = df_wide.loc[df_wide["ratio"].between(low_q, high_q),].reset_index(drop=True)
df_mid10.to_csv("../data/clean/ILO_wide_ratio_no_outliers.csv")

In [79]:
low_q, high_q = df_wide["ratio"].quantile([0.1, 0.9])

mask_outliers = ~df_wide["ratio"].between(low_q, high_q)
df_outliers = df_wide.loc[mask_outliers]

print(df_outliers.sort_values(by="ratio", ascending=True))

ool = df_outliers.sort_values(by="ratio", ascending=True)[
    ["country", "year", "ratio", "female_share", "balanced_ratio"]
]
ool.to_csv("../data/clean/ILO_wide_ratio_outliers.csv")

      ISO  year       country  break local_currency_code    F_local     F_usd  \
598   EGY  2009         Egypt  False                 EGP     79.912    14.413   
1802  TJK  2009    Tajikistan  False                 TJS    175.016    42.247   
1267  MLI  2018          Mali  False                 XOF  35629.446    64.146   
1355  NER  2011         Niger   True                 XOF  28459.135    60.391   
1644  SDN  2022         Sudan   True                 SDG  68495.762   125.276   
...   ...   ...           ...    ...                 ...        ...       ...   
1946  ZAF  2000  South Africa   True                 ZAR  36235.665  5221.406   
198   BHR  2019       Bahrain   True                 BHD    438.514  1166.260   
192   BHR  2007       Bahrain  False                 BHD    287.000   763.298   
1686  SLV  2020   El Salvador  False                 USD     54.933    54.933   
193   BHR  2008       Bahrain  False                 BHD    344.000   914.894   

         M_local     M_usd 

In [80]:
df_mid10.sort_values(by="ratio", ascending=True)

,ISO,year,country,break,local_currency_code,F_local,F_usd,M_local,M_usd,T_local,T_usd,ratio,female_share,male_share,balanced_avg,balanced_ratio
560,GBR,2014,United Kingdom of Great Britain and Northern I...,False,EUR,1639.382,2177.921,2536.357,3369.553,2086.503,2771.921,0.646,0.501,0.498,2087.869,0.785
256,CHE,2020,Switzerland,False,EUR,4674.540,5339.241,7233.639,8262.234,6009.231,6863.720,0.646,0.478,0.521,5954.089,0.785
893,LUX,2004,Luxembourg,False,EUR,2659.750,3308.469,4113.920,5117.314,3534.375,4396.418,0.646,0.398,0.601,3386.835,0.785
55,AUS,2006,Australia,False,AUD,2743.214,2065.719,4238.077,3191.395,3519.786,2650.501,0.647,0.480,0.519,3490.645,0.785
54,AUS,2002,Australia,False,AUD,2263.166,1229.607,3496.167,1899.512,2921.565,1587.323,0.647,0.466,0.533,2879.666,0.785
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1077,MYS,2014,Malaysia,False,MYR,2125.000,649.279,2232.000,681.972,2193.000,670.056,0.952,0.364,0.635,2178.500,0.975
159,BMU,2012,Bermuda,False,BMD,5732.200,5732.200,6015.600,6015.600,5869.800,5869.800,0.952,0.514,0.485,5873.900,0.975
1563,ZAF,2005,South Africa,False,ZAR,2816.281,442.858,2955.267,464.714,2907.479,457.199,0.952,0.343,0.656,2885.774,0.975
1432,THA,2013,Thailand,False,THB,13276.446,432.092,13940.771,453.713,13638.310,443.869,0.952,0.455,0.544,13608.608,0.975
